# Pretrain a small language model with TrainLM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Dhiraj309/TrainLM/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb)

This short native TorchTPU example trains GPT-2 on raw TinyStories text. `MODEL_INITIALIZATION="random"` builds its config without downloading model weights; set it to `"pretrained"` for continued pretraining. For tokenized datasets and custom model settings, see the [pretraining guide](../README.md#start-here-pretrain-a-language-model-from-scratch).

The 10,000-token budget is a workflow smoke run, not a quality result. This TPU v5e recipe uses BF16 and reports synchronized tokens/second, MFU when the runtime/model data supports it, and dataloader wait. Increase the token budget and compare later windows for useful performance measurements. Checkpoints and reports stay in `RUN_DIR`; publishing is a separate opt-in API call.


In [ ]:
import subprocess
import sys

source = "trainlm @ git+https://github.com/Dhiraj309/TrainLM.git@main"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", source], check=True)
requirements = ["huggingface-hub>=1.5,<2", "PyYAML>=6,<7", "transformers>=5.15,<6", "datasets>=3,<5", "tokenizers>=0.21,<1"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True)
from importlib.metadata import PackageNotFoundError, version
print("TrainLM installed without asking pip to replace PyTorch or TorchTPU.")
for package in ("torch-tpu", "libtpu"):
    try:
        print(f"{package}=={version(package)}")
    except PackageNotFoundError:
        print(f"{package}=MISSING; connect a runtime that provides TorchTPU")


In [ ]:
import json
from importlib.metadata import distribution
from trainlm import TrainLMPretrainingConfig, run_pretraining
source = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
source_revision = source.get("vcs_info", {}).get("commit_id")
assert source_revision, "Reinstall TrainLM from Git to record its exact commit."
RUN_DIR = "trainlm-pretraining"
FRESH_MAX_TOKENS, FRESH_MAX_TRAIN_EXAMPLES, LOGGING_STEPS = 10_000, 2_000, 10
MODEL_ID, MODEL_INITIALIZATION = "openai-community/gpt2", "random"
TOKENIZER_ID = MODEL_ID
DATASET_ID, DATASET_CONFIG_NAME, DATASET_REVISION = "roneneldan/TinyStories", None, "main"
TRAIN_SPLIT, VALIDATION_SPLIT = "train", "validation"
config = TrainLMPretrainingConfig(
    output_dir=RUN_DIR, max_tokens=FRESH_MAX_TOKENS, max_train_examples=FRESH_MAX_TRAIN_EXAMPLES,
    model_name_or_path=MODEL_ID, model_initialization=MODEL_INITIALIZATION,
    tokenizer_name_or_path=TOKENIZER_ID, dataset_source="hub", dataset_id=DATASET_ID,
    dataset_config_name=DATASET_CONFIG_NAME, dataset_revision=DATASET_REVISION,
    train_split=TRAIN_SPLIT, validation_split=VALIDATION_SPLIT, sequence_length=128,
    logging_steps=LOGGING_STEPS, accelerator="native_tpu", bf16=True,
)
print(json.dumps(config.training_plan(inspect_model=True), indent=2))
result = run_pretraining(config, show_progress=True, source_revision=source_revision)
print(result.summary())
print(f"checkpoint: {result.checkpoint_path} | report: {result.report_path}")
print(f"model: {result.export_path}")


In [ ]:
from dataclasses import replace
from trainlm import restore_pretraining_report, run_pretraining
RESUME_MODE = "skip"  # Choose "resume" or "extend" after the fresh run.
if RESUME_MODE in {"resume", "extend"}:
    saved = restore_pretraining_report(f"{RUN_DIR}/report.json")
    resume_config = saved.config
    extend_budget = RESUME_MODE == "extend"
    if extend_budget:
        extra_tokens = max(2_000, resume_config.max_tokens // 5)
        resume_config = replace(resume_config, max_tokens=resume_config.max_tokens + extra_tokens)
    options = {"resume_from_checkpoint": saved.checkpoint_path, "extend_training_budget": extend_budget}
    print(json.dumps(resume_config.training_plan(**options), indent=2))
    result = run_pretraining(
        resume_config, **options, show_progress=True, source_revision=source_revision,
        report_path=f"{RUN_DIR}/resume-report.json", export_path=f"{RUN_DIR}/resumed-export",
    )
    print(result.summary())
    print(f"resumed_from={saved.checkpoint_path} | report={result.report_path}")
else:
    print("Set RESUME_MODE to 'resume' or 'extend', then rerun this cell.")


In [ ]:
from trainlm import bundle_pretraining_artifacts
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
if hasattr(result, "export_path"):
    model_path = str(result.export_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
    prompt = tokenizer("Once upon a time", return_tensors="pt")
    with torch.inference_mode():
        generated = model.generate(**prompt, max_new_tokens=24, do_sample=False)
    print(tokenizer.decode(generated[0], skip_special_tokens=True))
bundle = bundle_pretraining_artifacts(result.report_path)
try:
    from google.colab import files
except ImportError:
    print(f"Artifact bundle: {bundle}")
else:
    files.download(str(bundle))
